# Initial Setup

In [ ]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog verilator gtkwave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 67.1 MB/s eta 0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,976 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.3 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-driv

In [ ]:
#@title Setting up PPA estimation

%%bash
set -e
cd /content

URL=$(python3 - <<'PY'
import json, urllib.request
data = json.load(urllib.request.urlopen(
    "https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest"
))
asset = next(
    a for a in data["assets"]
    if "linux-x64" in a["name"] and a["name"].endswith(".tgz")
)
print(asset["browser_download_url"])
PY
)

wget -q -O oss-cad-suite.tgz "$URL"
rm -rf /content/oss-cad-suite /content/oss-cad-suite-install
mkdir -p /content/oss-cad-suite-install
tar -xzf oss-cad-suite.tgz -C /content/oss-cad-suite-install
mv /content/oss-cad-suite-install/oss-cad-suite /content/oss-cad-suite

/content/oss-cad-suite/bin/yosys -V



Yosys 0.70+190 (git sha1 485b63564, Release, Clang /usr/bin/clang++ 21.1.8)


In [ ]:
#@title Select Model
#define the model to be used
model_choice = "gpt-5.2"
#model_choice = "gpt-5.2-codex"
#model_choice = "gpt-4o"
#model_choice = "claude-sonnet-4-6"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"
#model_choice = "nyu-dice-lab/VeriThoughts-Reasoning-14B" #path to your desired Hugging Face model

In [ ]:
#@title Utility functions

import sys
import os
import time
import re
from abc import ABC, abstractmethod

import openai
import anthropic

try:
    import google.genai.errors
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

################################################################################
### LOGGING
################################################################################

class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################

class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        openai.api_key = os.environ["OPENAI_API_KEY"]
        self.client = openai.OpenAI()
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [
            {
                "role": msg["role"] if msg["role"] in ["system", "user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )

        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20

        messages = [
            {
                "role": msg["role"] if msg["role"] in ["user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=messages
                ).content[0].text
                return output

            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)

        print(f"Failed, exceeded max retries {max_retries}")
        return ""


class Gemini(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        if genai is None:
            raise ImportError("google-genai is not installed or could not be imported.")
        self.gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text

        return output


class HuggingFace(AbstractLLM):
    """
    Local Hugging Face model backend.

    Downloads the selected model from the Hugging Face Hub using snapshot_download,
    then loads it with transformers. The model is cached and reused so it is not
    reloaded every time generate_verilog() is called.
    """

    _loaded_model_id = None
    _tokenizer = None
    _model = None
    _local_model_path = None

    def __init__(
        self,
        model_id=None,
        max_new_tokens=4096,
        temperature=0.2,
        top_p=0.95,
        load_in_4bit=False,
        trust_remote_code=True,
        strip_thinking=True,
    ):
        super().__init__()

        self.model_id = model_id or model_choice

        # If the user accidentally wrote "/org/model", convert it to "org/model".
        # Absolute local paths like "/content/my_model" are still allowed.
        if self.model_id.startswith("/") and not self.model_id.startswith("/content/"):
            self.model_id = self.model_id[1:]

        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.load_in_4bit = load_in_4bit
        self.trust_remote_code = trust_remote_code
        self.strip_thinking = strip_thinking

        self.hf_token = (
            os.environ.get("HF_TOKEN")
            or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
            or None
        )

        self.cache_dir = os.environ.get("HF_HOME", "/content/huggingface")

        self._ensure_model_loaded()

    def _get_torch_dtype(self):
        import torch

        if not torch.cuda.is_available():
            return torch.float32

        major, minor = torch.cuda.get_device_capability(0)

        # A100/H100 and newer generally support bf16 well.
        if major >= 8:
            return torch.bfloat16

        # T4/V100 are safer with fp16.
        return torch.float16

    def _ensure_model_loaded(self):
        if HuggingFace._model is not None and HuggingFace._loaded_model_id == self.model_id:
            return

        import torch
        from huggingface_hub import snapshot_download
        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[HuggingFace] Downloading/caching model: {self.model_id}")

        # If this is a local path, do not call snapshot_download.
        if os.path.isdir(self.model_id):
            local_model_path = self.model_id
        else:
            local_model_path = snapshot_download(
                repo_id=self.model_id,
                cache_dir=self.cache_dir,
                token=self.hf_token,
            )

        print(f"[HuggingFace] Loading tokenizer from: {local_model_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            local_model_path,
            trust_remote_code=self.trust_remote_code,
            token=self.hf_token,
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        torch_dtype = self._get_torch_dtype()

        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto",
            "low_cpu_mem_usage": True,
            "token": self.hf_token,
        }

        if torch.cuda.is_available() and self.load_in_4bit:
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=False,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch_dtype,
                bnb_4bit_use_double_quant=True,
            )

            model_kwargs["quantization_config"] = quantization_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        print("[HuggingFace] Loading model. This can take a few minutes the first time.")

        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            **model_kwargs
        )

        model.eval()

        HuggingFace._loaded_model_id = self.model_id
        HuggingFace._tokenizer = tokenizer
        HuggingFace._model = model
        HuggingFace._local_model_path = local_model_path

        print("[HuggingFace] Model loaded successfully.")

    def _format_prompt(self, conversation: Conversation):
        tokenizer = HuggingFace._tokenizer

        messages = []
        for msg in conversation.get_messages():
            role = msg.get("role", "user")
            content = msg.get("content", "")

            if role not in ["system", "user", "assistant"]:
                role = "user"

            messages.append({"role": role, "content": content})

        # Prefer the model's own chat template when available.
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            # Generic fallback for non-chat-template models.
            prompt_parts = []
            for msg in messages:
                prompt_parts.append(f"{msg['role'].upper()}:\n{msg['content']}\n")
            prompt_parts.append("ASSISTANT:\n")
            return "\n".join(prompt_parts)

    def _clean_output(self, text):
        text = text.strip()

        # Helpful for reasoning models that emit <think>...</think>.
        if self.strip_thinking:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

        return text

    def generate(self, conversation: Conversation, num_choices=1):
        import torch

        tokenizer = HuggingFace._tokenizer
        model = HuggingFace._model

        prompt = self._format_prompt(conversation)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=False,
        )

        # With device_map="auto", use the first model parameter's device.
        first_device = next(model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}

        do_sample = self.temperature is not None and self.temperature > 0

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=do_sample,
                temperature=self.temperature if do_sample else None,
                top_p=self.top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        generated_ids = outputs[0][prompt_len:]

        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return self._clean_output(text)


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################

def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No Verilog modules found in response")
        exit(3)

    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type=None, model_id=""):
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    selected_model_id = model_id or model_choice

    if model_type == "ChatGPT":
        model = ChatGPT(model_id=selected_model_id)
    elif model_type == "Claude":
        model = Claude(model_id=selected_model_id)
    elif model_type == "Gemini":
        model = Gemini(model_id=selected_model_id)
    elif model_type == "HuggingFace":
        model = HuggingFace(model_id=selected_model_id)
    else:
        raise ValueError(
            f"Invalid model type: {model_type}. "
            "Expected one of: ChatGPT, Claude, Gemini, HuggingFace."
        )

    return model.generate(conv)

In [ ]:
#@title Feedback Loop

import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []
    for p in file_list:
        if not p:
            continue

        try:
            with open(p, "r") as f:
                text = f.read()
                buf.append(text)
                if not text.endswith("\n"):
                    buf.append("\n")
        except FileNotFoundError:
            print(f"[Warning] Previous module file not found and will be skipped: {p}")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the end of long logs, where compiler errors usually appear."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    tail = s[-max_chars:]
    return "[...truncated...]\n" + tail


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a way that works for API models and local HF models.

    ChatGPT/Gemini/HuggingFace can use a system role.
    Claude path in the earlier utility cell maps unsupported roles to user,
    but we keep Claude as user here to avoid role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for your Verilog testbenches.

    Accepts:
      - "All tests passed"
      - "passed!"
      - "Test x passed"

    Rejects:
      - "failed"
      - "Some tests failed"
      - non-empty fatal/error-like stderr
    """
    out = (stdout_text or "")
    err = (stderr_text or "")
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "error running",
        "fatal",
        "syntax error",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    outdir="",
    log=None,
    prev_modules=None,
    model_id="",
):
    """
    Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Key behaviors:
      - The LLM is asked to output ONLY the current module.
      - For compilation, this concatenates:
            [prev_modules...] + [current module]
        into a temporary combined file.
      - After an error, repair iterations only resend:
            (a) previous module version
            (b) most recent compiler/simulation feedback
        to keep prompt size controlled.
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]
    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    if outdir is None:
        outdir = ""

    outdir = outdir.strip()

    if outdir != "":
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir == "":
            return filename
        return os.path.join(outdir, filename)

    # Normalize prev_modules into a list of paths.
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read the testbench so the model can target the test expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is incorrect or has a compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete corrected Verilog module, from module declaration through endmodule. "
        "Use Verilog-2001 only unless explicitly told otherwise."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench and design prompt."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it. Only use it to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: only previous module plus most recent error feedback."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module. "
            "Do not output explanations. "
            "Do not output a testbench. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)
        return conv

    conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    module_file = _outpath(module + ".v")
    combined_file = _outpath(module + "__combined.v")
    sim_output_file = _outpath(module)

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        print(f"\n========== Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        # Generate or repair only the current module.
        response = generate_verilog(conv, model_type, model_id=model_id)

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        conv.add_message("assistant", response)

        # Write raw model response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_" + str(iterations) + ".txt")
        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and write just the current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            last_module_text = response
            conv = _new_conv_repair_only(last_module_text, last_error_text)

            if iterations >= max_iterations:
                timeout = True

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)
            continue

        # Build concatenated source for compilation:
        # previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(module_file, "r") as f:
            curr_text = f.read()

        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)
            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # Compile with iVerilog.
        compile_cmd = [
            "iverilog",
            "-o",
            sim_output_file,
            combined_file,
            testbench,
        ]

        proc = subprocess.run(
            compile_cmd,
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            last_error_text = (
                "iverilog failed.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        elif proc.stderr.strip() != "":
            # Treat warnings as repair feedback.
            # This keeps generated modules cleaner for later synthesis/PPA.
            status = "Warnings compiling testbench"
            print(status)

            last_error_text = (
                "iverilog produced warnings.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        else:
            # Run simulation.
            run_cmd = ["vvp", sim_output_file]

            proc = subprocess.run(
                run_cmd,
                capture_output=True,
                text=True,
            )

            result_stdout = proc.stdout or ""
            result_stderr = proc.stderr or ""

            print(result_stdout)

            if proc.returncode != 0:
                status = "Error running testbench"
                print(status)

                last_error_text = (
                    "vvp simulation failed.\n\n"
                    "Command:\n"
                    + " ".join(run_cmd)
                    + "\n\nstdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            elif not _simulation_passed(result_stdout, result_stderr):
                status = "Testbench did not report success"
                print(status)

                last_error_text = (
                    "Simulation completed, but the output did not indicate success.\n\n"
                    "stdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            else:
                status = "Testbench ran successfully"
                print(status)

                last_error_text = ""
                success = True

        # Write bounded iteration log.
        with open(_outpath("log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

            if last_error_text:
                file.write("\nMost recent feedback:\n")
                file.write(_truncate_text(last_error_text))

        if not success:
            # Keep future repair prompts small.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("\n========== Feedback Loop Complete ==========")
    print("Final status: ", status)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return (
        np.sum(timelist_total),
        np.sum(timelist_gen),
        np.sum(timelist_error),
    )

In [ ]:
#@title Hierarchical Loop

import os
import re
import numpy as np


def _normalize_to_list(x):
    """Normalize None/string/list/tuple into a list."""
    if x is None or x == "":
        return []
    if isinstance(x, str):
        return [x]
    return list(x)


def _resolve_prev_module_path(item):
    """
    Resolve previous module reference into a .v path.

    Supported forms:
      - "mux2_1"              -> "./mux2_1/mux2_1.v"
      - "./mux2_1/mux2_1.v"   -> "./mux2_1/mux2_1.v"
      - ("mux2_1", "...", ...) -> "./mux2_1/mux2_1.v"
    """
    if isinstance(item, (list, tuple)):
        item = item[0]

    item = str(item)

    if item.endswith(".v"):
        return item

    return os.path.join("./" + item, item + ".v")


def _extract_verilog_interfaces(verilog_text):
    """
    Extract module declarations/interfaces from Verilog text.

    This is useful for hierarchical prompting because the current module usually
    only needs previous module names and port lists, not full implementations.
    """
    if not verilog_text:
        return ""

    # Match:
    #   module name (...);
    #   module name #(parameters...) (...);
    pattern = r'\bmodule\s+[A-Za-z_][A-Za-z0-9_]*\s*(?:#\s*\(.*?\)\s*)?\(.*?\)\s*;'

    matches = re.findall(pattern, verilog_text, flags=re.DOTALL)

    if not matches:
        return ""

    cleaned = []
    for m in matches:
        cleaned.append(m.strip() + "\nendmodule")

    return "\n\n".join(cleaned) + "\n"


def _read_prev_modules_for_prompt(prev_verilog_files, context_mode="interfaces", max_chars=24000):
    """
    Read previous modules for LLM context.

    context_mode:
      - "interfaces": include only module declarations, best for HF/local models.
      - "full": include full previous module bodies.
      - "none": include no previous code in the prompt.

    Compilation still receives the full previous .v files through verilog_loop(prev_modules=...).
    """
    if context_mode == "none":
        return ""

    chunks = []

    for p in prev_verilog_files:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found for prompt context: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        if context_mode == "interfaces":
            txt = _extract_verilog_interfaces(txt)

        if txt.strip():
            chunks.append("// File: " + p + "\n" + txt.strip() + "\n")

    prev_text = "\n".join(chunks)

    if len(prev_text) > max_chars:
        # Keep the most recent context, because later modules tend to depend on recent modules.
        prev_text = "[...previous module context truncated...]\n" + prev_text[-max_chars:]

    return prev_text


def _call_verilog_loop_compatible(
    prompt,
    module,
    testbench,
    max_iterations,
    model,
    outdir,
    log,
    prev_verilog_files,
    model_id="",
):
    """
    Calls the updated verilog_loop with model_id support.
    Falls back to the older signature if needed.
    """
    try:
        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
            model_id=model_id,
        )
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] verilog_loop does not accept model_id. Falling back to old signature.")

        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
        )


def hier_gen(
    submods,
    prevsubmods="",
    max_iterations=10,
    model_type=None,
    model_id="",
    testbench_template="./{module}tb.v",
    prompt_prev_context_chars=24000,
    prev_context_mode="interfaces",
):
    """
    Hierarchically generate a design by building submodules in order.

    submods:
      list of tuples/lists:
        [
          (file_basename, human_name, io_string),
          ...
        ]

      Example:
        [
          ("mux2_1", "2-to-1 mux", "input wire a, input wire b, input wire sel, output wire y"),
          ("mux4_1", "4-to-1 mux", "input wire [3:0] in, input wire [1:0] sel, output wire y"),
        ]

    prevsubmods:
      Optional previous module references already generated before this call.

      Supported:
        - ""
        - None
        - "mux2_1"
        - ["mux2_1", "mux4_1"]
        - ["./mux2_1/mux2_1.v"]
        - previous submod tuples/lists

    model_type:
      Optional. If None, uses os.environ["MODEL"].

      Supported by the earlier utility cell:
        - "ChatGPT"
        - "Claude"
        - "Gemini"
        - "HuggingFace"

    model_id:
      Optional model override passed through to verilog_loop/generate_verilog.

      Example:
        model_id="nyu-dice-lab/VeriThoughts-Reasoning-14B"

    testbench_template:
      Default assumes testbenches are named:
        ./module_nametb.v

      You can also use:
        "./testbenches/{module}_tb.v"

    prev_context_mode:
      - "interfaces": best default, includes previous module port declarations only.
      - "full": includes full previous module code in prompt.
      - "none": does not include previous modules in prompt.

      Regardless of this setting, previous modules are still passed to iVerilog
      through verilog_loop(prev_modules=...).
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_context_modes = ["interfaces", "full", "none"]
    if prev_context_mode not in valid_context_modes:
        raise ValueError(
            f"Invalid prev_context_mode: {prev_context_mode}. "
            f"Expected one of: {valid_context_modes}"
        )

    totaltime = []
    gentime = []
    errortime = []

    prev_verilog_files = []

    # Add previously generated external modules, if any.
    for s in _normalize_to_list(prevsubmods):
        p = _resolve_prev_module_path(s)
        if os.path.exists(p):
            prev_verilog_files.append(p)
        else:
            print(f"[Warning] prevsubmod file does not exist and will be skipped: {p}")

    if not submods:
        print("[Warning] No submodules provided.")
        return (0, 0, 0)

    overall = submods[-1][1]

    for i in range(len(submods)):
        fcurr = submods[i][0]   # file/module basename
        curr = submods[i][1]    # human-readable name
        iocurr = submods[i][2]  # port list string

        outdir = "./" + fcurr
        os.makedirs(outdir, exist_ok=True)

        prev_text = _read_prev_modules_for_prompt(
            prev_verilog_files,
            context_mode=prev_context_mode,
            max_chars=prompt_prev_context_chars,
        )

        if i == 0 and len(prev_verilog_files) == 0:
            prompt = (
                "// We will be generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// IMPORTANT: Output ONLY the module requested below.\n"
                "// Do NOT generate any previous modules, future modules, explanations, or testbenches.\n"
                "// Use Verilog-2001 only.\n"
                "// Begin by generating: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )
        else:
            prompt = (
                "// We are generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// Use Verilog-2001 only.\n\n"
                "// Previously generated modules are shown below for reference only.\n"
                "// You may instantiate them, but you must NOT repeat their definitions.\n\n"
                + prev_text
                + "\n"
                "// IMPORTANT: Output ONLY the new module requested below.\n"
                "// Do NOT repeat any previous modules.\n"
                "// Do NOT generate explanations, Markdown fences, or a testbench.\n"
                "// Now generate: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )

        module = fcurr
        testbench = testbench_template.format(module=fcurr, name=fcurr)
        model = model_type
        log = os.path.join(outdir, "log.txt")

        if not os.path.exists(testbench):
            print(f"[Warning] Testbench file does not exist yet: {testbench}")
            print("[Warning] verilog_loop will still run, but iVerilog will fail unless the file exists.")

        total, gen, error = _call_verilog_loop_compatible(
            prompt=prompt,
            module=module,
            testbench=testbench,
            max_iterations=max_iterations,
            model=model,
            outdir=outdir,
            log=log,
            prev_verilog_files=prev_verilog_files,
            model_id=model_id,
        )

        print("Module ", i + 1, " done: ", curr)

        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)

        # Add the current module file to the compilation context for subsequent modules.
        curr_module_file = os.path.join(outdir, module + ".v")

        if os.path.exists(curr_module_file):
            prev_verilog_files.append(curr_module_file)
        else:
            print(f"[Warning] Expected generated module file not found: {curr_module_file}")

    overall_total = np.sum(totaltime)
    overall_gen = np.sum(gentime)
    overall_error = np.sum(errortime)

    print("Overall Total time: ", overall_total)
    print("Overall Generation Time: ", overall_gen)
    print("Overall Error handling time: ", overall_error)

    return (overall_total, overall_gen, overall_error)

In [ ]:
#@title PPA Optimizer Loop

#@title PPA Optimization Loop (Yosys no-liberty + iVerilog correctness)

import subprocess
import sys
import os
import time
import math
import json
import re
import shutil
import numpy as np


################################################################################
### GENERAL HELPERS
################################################################################

def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []

    for p in file_list:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found and will be skipped: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        buf.append(txt)

        if txt and not txt.endswith("\n"):
            buf.append("\n")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the tail of long logs."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    return "[...truncated...]\n" + s[-max_chars:]


def _shell_quote_for_yosys(path):
    """Quote filenames for use inside a .ys script."""
    return '"' + str(path).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=None):
    try:
        return int(x)
    except Exception:
        return default


def _first_regex(text, patterns, cast=float):
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            try:
                return cast(m.group(1))
            except Exception:
                pass

    return None


def _normalize_clk_signal(clk_signal):
    """Treat None, '', 'None', 'none', etc. as no clock."""
    if clk_signal is None:
        return None

    s = str(clk_signal).strip()

    if s == "" or s.lower() == "none":
        return None

    return s


def _has_any_parsed_metric(metrics):
    if not metrics:
        return False

    keys = [
        "transistor_est",
        "area_cmos",
        "num_cells",
        "num_wire_bits",
        "timing_proxy",
    ]

    return any(metrics.get(k) is not None for k in keys)


def _is_verbose(verbose):
    return str(verbose).strip().lower() in ["1", "true", "yes", "y"]


def _write_text(path, text):
    with open(path, "w") as f:
        f.write(text)


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a provider-compatible way.

    ChatGPT, Gemini, and HuggingFace can use the system role through the
    Conversation abstraction from the previous utility cell.

    Claude is kept as user to avoid unsupported-role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _generate_verilog_compatible(conv, model_type, model_id=""):
    """
    Calls generate_verilog with model_id support when available.
    Falls back to the older two-argument signature if needed.
    """
    try:
        return generate_verilog(conv, model_type, model_id=model_id)
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] generate_verilog does not accept model_id. Falling back to old signature.")
        return generate_verilog(conv, model_type)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for Verilog testbenches.

    Accepts:
      - All tests passed
      - passed!
      - Test x passed
      - Test x passed!

    Rejects:
      - failed
      - Some tests failed
      - fatal/error markers
    """
    out = stdout_text or ""
    err = stderr_text or ""
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "fatal",
        "syntax error",
        "error:",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


################################################################################
### YOSYS PARSING
################################################################################

def _parse_yosys_stat_text(stat_text):
    """
    Parse Yosys `stat -tech cmos` output.

    Supports both styles:
      Older style:
        Number of wires: ...
        Number of cells: ...
        Chip area for module ...: ...

      Newer compact style:
        6 wires
        165 wire bits
        61 cells
        Estimated number of transistors: 722
    """
    metrics = {}

    metrics["num_wires"] = _first_regex(
        stat_text,
        [
            r"Number of wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wires\s*$",
        ],
        int,
    )

    metrics["num_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wire bits\s*$",
        ],
        int,
    )

    metrics["num_public_wires"] = _first_regex(
        stat_text,
        [
            r"Number of public wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wires\s*$",
        ],
        int,
    )

    metrics["num_public_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of public wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wire bits\s*$",
        ],
        int,
    )

    metrics["num_memories"] = _first_regex(
        stat_text,
        [
            r"Number of memories:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memories\s*$",
        ],
        int,
    )

    metrics["num_memory_bits"] = _first_regex(
        stat_text,
        [
            r"Number of memory bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memory bits\s*$",
        ],
        int,
    )

    metrics["num_processes"] = _first_regex(
        stat_text,
        [
            r"Number of processes:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+processes\s*$",
        ],
        int,
    )

    metrics["num_cells"] = _first_regex(
        stat_text,
        [
            r"Number of cells:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+cells\s*$",
        ],
        int,
    )

    transistor_est = _first_regex(
        stat_text,
        [
            r"Estimated number of transistors:\s*([0-9]+(?:\.[0-9]+)?)",
            r"Chip area for module .*?:\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["transistor_est"] = transistor_est

    # Backward-compatible area-like proxy.
    metrics["area_cmos"] = transistor_est

    return metrics


def _parse_yosys_timeest_text(time_text):
    """
    Parse a timing proxy from Yosys timeest output.

    Returns None if no timing proxy can be found.
    """
    metrics = {}

    if re.search(r"No domain .* in module", time_text or "", flags=re.IGNORECASE):
        metrics["timing_proxy"] = None
        return metrics

    timing_proxy = _first_regex(
        time_text,
        [
            r"critical path.*?([0-9]+(?:\.[0-9]+)?)\s*AIG",
            r"([0-9]+(?:\.[0-9]+)?)\s*AIG nodes",
            r"([0-9]+(?:\.[0-9]+)?)\s*nodes.*critical path",
            r"Estimated.*?([0-9]+(?:\.[0-9]+)?)",
            r"Delay\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["timing_proxy"] = timing_proxy

    return metrics


def _format_metrics(metrics):
    if metrics is None:
        return "None"

    parts = []

    if metrics.get("transistor_est") is not None:
        parts.append("transistor_est={:.0f}".format(metrics["transistor_est"]))
    elif metrics.get("area_cmos") is not None:
        parts.append("area_cmos={:.3f}".format(metrics["area_cmos"]))

    if metrics.get("timing_proxy") is not None:
        parts.append("timing_proxy={:.3f}".format(metrics["timing_proxy"]))

    if metrics.get("num_cells") is not None:
        parts.append("num_cells={}".format(metrics["num_cells"]))

    if metrics.get("num_wire_bits") is not None:
        parts.append("wire_bits={}".format(metrics["num_wire_bits"]))

    return ", ".join(parts) if parts else "No parsed metrics"


################################################################################
### YOSYS ESTIMATION
################################################################################

def yosys_estimate_no_liberty(verilog_file, top_module, clk_signal=None, outdir=""):
    """
    Run a coarse no-liberty Yosys estimate.

    Uses:
      - stat -top <top> -tech cmos
      - optional timeest -clk <clk>

    Returns:
      {
        "ok": bool,
        "metrics": dict,
        "stat_text": str,
        "time_text": str,
        "yosys_stdout": str,
        "yosys_stderr": str,
        "error": str
      }
    """
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    clk_signal = _normalize_clk_signal(clk_signal)

    script_path = os.path.join(outdir, "ppa_estimate.ys") if outdir else "ppa_estimate.ys"
    stat_path = os.path.join(outdir, "ppa_stat.txt") if outdir else "ppa_stat.txt"
    time_path = os.path.join(outdir, "ppa_time.txt") if outdir else "ppa_time.txt"

    ys = []
    ys.append("read_verilog {}".format(_shell_quote_for_yosys(verilog_file)))
    ys.append("hierarchy -check -top {}".format(top_module))
    ys.append("proc")
    ys.append("opt")
    ys.append("memory")
    ys.append("opt")
    ys.append("fsm")
    ys.append("opt")
    ys.append("techmap")
    ys.append("opt")
    ys.append("clean")
    ys.append(
        "tee -o {} stat -top {} -tech cmos".format(
            _shell_quote_for_yosys(stat_path),
            top_module,
        )
    )

    if clk_signal is not None:
        ys.append("flatten")
        ys.append(
            "tee -o {} timeest -clk {}".format(
                _shell_quote_for_yosys(time_path),
                clk_signal,
            )
        )

    with open(script_path, "w") as f:
        f.write("\n".join(ys) + "\n")

    proc = subprocess.run(
        ["yosys", "-s", script_path],
        capture_output=True,
        text=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    result = {
        "ok": False,
        "metrics": {},
        "stat_text": "",
        "time_text": "",
        "yosys_stdout": stdout,
        "yosys_stderr": stderr,
        "error": "",
    }

    if proc.returncode != 0:
        result["error"] = "Yosys failed.\nstdout:\n{}\nstderr:\n{}".format(stdout, stderr)
        return result

    stat_text = ""
    time_text = ""

    if os.path.exists(stat_path):
        with open(stat_path, "r") as f:
            stat_text = f.read()

    if os.path.exists(time_path):
        with open(time_path, "r") as f:
            time_text = f.read()

    metrics = _parse_yosys_stat_text(stat_text)
    metrics.update(_parse_yosys_timeest_text(time_text))

    if metrics.get("transistor_est") is not None:
        metrics["power_proxy"] = metrics["transistor_est"]
    else:
        metrics["power_proxy"] = metrics.get("num_cells")

    result["ok"] = True
    result["metrics"] = metrics
    result["stat_text"] = stat_text
    result["time_text"] = time_text

    if not _has_any_parsed_metric(metrics):
        result["error"] = "Yosys ran successfully, but no metrics were parsed."

    return result


################################################################################
### PPA SCORING
################################################################################

def _component_ratio(candidate_value, baseline_value):
    if candidate_value is None or baseline_value is None:
        return None

    if baseline_value == 0:
        return None

    return float(candidate_value) / float(baseline_value)


def _relative_ppa_score(
    candidate_metrics,
    baseline_metrics,
    area_weight=0.60,
    timing_weight=0.30,
    power_weight=0.10,
):
    """
    Lower is better.

    Uses relative ratios against the current best correct design.
    """
    used_weight = 0.0
    score = 0.0

    cand_area = candidate_metrics.get("transistor_est", candidate_metrics.get("area_cmos"))
    base_area = baseline_metrics.get("transistor_est", baseline_metrics.get("area_cmos"))

    area_ratio = _component_ratio(cand_area, base_area)

    if area_ratio is not None:
        score += area_weight * area_ratio
        used_weight += area_weight

    timing_ratio = _component_ratio(
        candidate_metrics.get("timing_proxy"),
        baseline_metrics.get("timing_proxy"),
    )

    if timing_ratio is not None:
        score += timing_weight * timing_ratio
        used_weight += timing_weight

    power_ratio = _component_ratio(
        candidate_metrics.get("power_proxy"),
        baseline_metrics.get("power_proxy"),
    )

    if power_ratio is not None:
        score += power_weight * power_ratio
        used_weight += power_weight

    if used_weight == 0.0:
        return math.inf

    return score / used_weight


def _is_better_ppa(candidate_metrics, best_metrics, min_improvement=0.0025):
    """
    Returns True only when candidate has parsed metrics and is better.
    """
    if not _has_any_parsed_metric(candidate_metrics):
        return False

    if best_metrics is None or not _has_any_parsed_metric(best_metrics):
        return True

    cand_score = _relative_ppa_score(candidate_metrics, best_metrics)

    # Best-vs-itself is effectively 1.0, so lower is better.
    return cand_score < (1.0 - min_improvement)


################################################################################
### IVERILOG CHECK
################################################################################

def _compile_and_run_iverilog(module_name, combined_file, testbench, outdir):
    """
    Return:
      {
        "ok": bool,
        "status": str,
        "error_text": str,
        "stdout": str,
        "stderr": str
      }
    """
    exe_path = os.path.join(outdir, module_name) if outdir else module_name

    proc = subprocess.run(
        ["iverilog", "-o", exe_path, combined_file, testbench],
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        return {
            "ok": False,
            "status": "Error compiling testbench",
            "error_text": (
                "iverilog failed.\n"
                "Command:\n"
                + " ".join(["iverilog", "-o", exe_path, combined_file, testbench])
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            ),
            "stdout": proc.stdout or "",
            "stderr": proc.stderr or "",
        }

    warn_text = ""

    if proc.stderr and proc.stderr.strip():
        warn_text = (
            "iverilog produced warnings.\n"
            "stderr:\n"
            + (proc.stderr or "")
            + "\nstdout:\n"
            + (proc.stdout or "")
        )

    sim = subprocess.run(
        ["vvp", exe_path],
        capture_output=True,
        text=True,
    )

    result_stdout = sim.stdout or ""
    result_stderr = sim.stderr or ""

    if sim.returncode != 0:
        return {
            "ok": False,
            "status": "Error running testbench",
            "error_text": (
                "vvp simulation failed.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    if not _simulation_passed(result_stdout, result_stderr):
        return {
            "ok": False,
            "status": "Testbench did not report success",
            "error_text": (
                "Simulation completed, but the output did not indicate success.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    return {
        "ok": True,
        "status": "Testbench ran successfully",
        "error_text": warn_text,
        "stdout": result_stdout,
        "stderr": result_stderr,
    }


################################################################################
### MAIN PPA LOOP
################################################################################

def ppa_verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    clk_signal=None,
    outdir="",
    log=None,
    prev_modules=None,
    verbose="0",
    model_id="",
    keep_best_as_module=True,
):
    """
    Generate functionally correct Verilog, then iteratively improve coarse no-liberty
    PPA using Yosys estimates while preserving correctness.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Assumptions:
      - generate_verilog(conv, model_type, model_id=...) exists from the utility cell
      - Conversation(log_file=...) exists
      - write_code_blocks_to_file(response, module, path) exists
      - yosys, iverilog, and vvp are in PATH

    Strategy:
      1. Generate a candidate.
      2. Check correctness with iVerilog/vvp.
      3. If correct, estimate PPA with Yosys.
      4. Keep the best correct candidate seen so far.
      5. Ask the LLM to improve the best correct candidate further.
      6. If a candidate fails, repair using only the failed module plus latest error.
    """
    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]

    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    outdir = "" if outdir is None else str(outdir).strip()

    if outdir:
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir:
            return os.path.join(outdir, filename)
        return filename

    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    tb_text = ""

    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog-2001 code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete Verilog module from module declaration through endmodule. "
        "Preserve the exact module name and port interface. "
        "Preserve functionality required by the testbench. "
        "Optimize for lower coarse PPA under Yosys without a liberty file: "
        "reduce logic depth, reduce unnecessary cells, share repeated logic, simplify control, "
        "avoid redundant muxing, avoid unnecessary intermediate signals, and keep widths disciplined."
    )

    def _new_conv_initial():
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile and simulate your module. "
                "Do NOT generate a testbench or modify it. Use it only to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n"
            )

        conv.add_message(
            "user",
            design_prompt
            + "\n\n"
            "Also optimize the module for better coarse Yosys-estimated PPA without a liberty file. "
            "Output ONLY the full Verilog-2001 module. "
            "Do not use Markdown fences."
        )

        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog-2001 module so it compiles and passes the testbench. "
            "Preserve the exact interface and intended behavior. "
            "Keep PPA reasonably optimized, but correctness is the priority for this step. "
            "Output ONLY the full corrected module. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation/Yosys output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    def _new_conv_optimize_from_best(best_module_text, best_metrics, last_candidate_metrics=None):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Starting from the following functionally correct Verilog-2001 module, "
            "produce a new candidate that preserves the exact same behavior and interface "
            "but improves coarse Yosys-estimated PPA without a liberty file.\n\n"
            "Current best metrics:\n"
            + _format_metrics(best_metrics)
            + "\n\n"
        )

        if last_candidate_metrics is not None:
            prompt += (
                "Most recent candidate metrics:\n"
                + _format_metrics(last_candidate_metrics)
                + "\n\n"
            )

        prompt += (
            "Optimization priorities:\n"
            "1. Lower stat -tech cmos transistor_est / area proxy\n"
            "2. Lower timing proxy, if available\n"
            "3. Fewer cells and wire bits\n\n"
            "Rules:\n"
            "- Do not add helper modules.\n"
            "- Do not change functionality.\n"
            "- Do not change the module name or port interface.\n"
            "- Use Verilog-2001 only.\n"
            "- Output ONLY the full module.\n"
            "- Do not use Markdown fences.\n\n"
            "Current best module:\n"
            "```verilog\n"
            + best_module_text
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    # Timing
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    # Files
    module_file = _outpath(module + ".v")
    best_module_file = _outpath(module + "_best.v")
    combined_file = _outpath(module + "__combined.v")

    # Best-so-far state
    best_module_text = None
    best_metrics = None
    best_iter = None

    # Latest state
    conv = _new_conv_initial()
    last_error_text = ""
    last_candidate_metrics = None

    iterations = 0
    timeout = False

    while not timeout:
        print(f"\n========== PPA Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        response = _generate_verilog_compatible(
            conv,
            model_type,
            model_id=model_id,
        )

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        try:
            conv.add_message("assistant", response)
        except Exception:
            pass

        # Save raw response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_{:03d}.txt".format(iterations))

        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and persist current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            conv = _new_conv_repair_only(response, last_error_text)

            candidate_metrics = None

            log_path = _outpath("log_iter_{:03d}.txt".format(iterations))
            with open(log_path, "w") as f:
                f.write("Iteration status: {}\n".format(status))
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)

            if iterations >= max_iterations:
                timeout = True

            continue

        with open(module_file, "r") as f:
            curr_text = f.read()

        # Concatenate previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(combined_file, "w") as f:
            f.write(prev_text)

            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")

            f.write(curr_text)

            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # 1. Functional check with iVerilog/vvp.
        iverilog_result = _compile_and_run_iverilog(
            module_name=module,
            combined_file=combined_file,
            testbench=testbench,
            outdir=outdir,
        )

        status = iverilog_result["status"]
        accepted = False
        candidate_metrics = None

        if not iverilog_result["ok"]:
            last_error_text = iverilog_result["error_text"]
            conv = _new_conv_repair_only(curr_text, last_error_text)

        else:
            # 2. Coarse PPA estimate with Yosys.
            ppa_dir = _outpath("iter_{:03d}_ppa".format(iterations))
            os.makedirs(ppa_dir, exist_ok=True)

            ppa_result = yosys_estimate_no_liberty(
                verilog_file=combined_file,
                top_module=module,
                clk_signal=clk_signal,
                outdir=ppa_dir,
            )

            if _is_verbose(verbose):
                print("YOSYS OK:", ppa_result["ok"])
                print("YOSYS STDOUT HEAD:\n", (ppa_result["yosys_stdout"] or "")[:3000])
                print("YOSYS STDERR HEAD:\n", (ppa_result["yosys_stderr"] or "")[:3000])
                print("STAT TEXT HEAD:\n", repr((ppa_result["stat_text"] or "")[:1000]))
                print("TIME TEXT HEAD:\n", repr((ppa_result["time_text"] or "")[:1000]))
                print("PARSED METRICS:", ppa_result["metrics"])

            if not ppa_result["ok"]:
                status = "Yosys estimation failed"
                last_error_text = ppa_result["error"]
                conv = _new_conv_repair_only(curr_text, last_error_text)

            else:
                candidate_metrics = ppa_result["metrics"]
                last_candidate_metrics = candidate_metrics

                if not _has_any_parsed_metric(candidate_metrics):
                    status = "Yosys ran but returned no parsable metrics"

                    if best_module_text is not None and best_metrics is not None:
                        conv = _new_conv_optimize_from_best(
                            best_module_text=best_module_text,
                            best_metrics=best_metrics,
                            last_candidate_metrics=None,
                        )
                    else:
                        conv = _new_conv_initial()

                elif _is_better_ppa(candidate_metrics, best_metrics):
                    best_module_text = curr_text
                    best_metrics = candidate_metrics
                    best_iter = iterations
                    accepted = True

                    _write_text(best_module_file, best_module_text)

                    status = "Accepted new best candidate"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text,
                        best_metrics=best_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

                else:
                    status = "Candidate correct but did not improve best PPA"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text if best_module_text is not None else curr_text,
                        best_metrics=best_metrics if best_metrics is not None else candidate_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

        # Write iteration log.
        log_path = _outpath("log_iter_{:03d}.txt".format(iterations))

        with open(log_path, "w") as f:
            try:
                f.write("\n".join(str(i) for i in conv.get_messages()))
            except Exception:
                f.write("(Conversation logging unavailable)\n")

            f.write("\n\nIteration status: {}\n".format(status))

            if candidate_metrics is not None:
                f.write("Candidate metrics: {}\n".format(_format_metrics(candidate_metrics)))

            if best_metrics is not None:
                f.write("Best metrics so far: {}\n".format(_format_metrics(best_metrics)))
                f.write("Best iteration so far: {}\n".format(best_iter))

            if last_error_text:
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

        print("Iteration {}: {}".format(iterations, status))

        if candidate_metrics is not None:
            print("  Candidate metrics:", _format_metrics(candidate_metrics))

        if best_metrics is not None:
            print("  Best metrics so far:", _format_metrics(best_metrics))
            print("  Best iteration so far:", best_iter)

        iterations += 1

        end_time = time.time()

        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

        if iterations >= max_iterations:
            timeout = True

    # Optional: leave module.v as the best candidate, not the last failed/non-improved candidate.
    if keep_best_as_module and best_module_text is not None:
        _write_text(module_file, best_module_text)
        print(f"Best candidate copied to: {module_file}")

    print("\n========== PPA Optimization Complete ==========")
    print("Best file: ", best_module_file if best_module_text is not None else None)
    print("Best metrics: ", _format_metrics(best_metrics) if best_metrics is not None else None)
    print("Best iteration: ", best_iter)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return {
        "best_file": best_module_file if best_module_text is not None else None,
        "best_metrics": best_metrics,
        "best_iteration": best_iter,
        "iterations": iterations,
        "total_time": float(np.sum(timelist_total)),
        "generation_time": float(np.sum(timelist_gen)),
        "error_handling_time": float(np.sum(timelist_error)),
    }

# Setting the API Key

In [ ]:
### Defaults to Claude, just change the model in the Select Model cell in Initial Setup & choose your provider here
### Or, choose a Hugging Face model and set "MODEL" to "HuggingFace"

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own API key as a string here:
os.environ["OPENAI_API_KEY"] = ""
os.environ['CLAUDE_API_KEY'] = "API key here"
os.environ['GEMINI_API_KEY'] ="API key here"
os.environ["MODEL"] = "ChatGPT"
#os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"
#os.environ["MODEL"] = "HuggingFace"

#Mux Hierarchy Example

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/mux/mux2_1tb.v ./mux2_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux4_1tb.v ./mux4_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux8_1tb.v ./mux8_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux16_1tb.v ./mux16_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux32_1tb.v ./mux32_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux64_1tb.v ./mux64_1tb.v

Cloning into 'ROME-LLM'...
remote: Enumerating objects: 444, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 444 (delta 114), reused 41 (delta 19), pack-reused 220 (from 1)
Receiving objects: 100% (444/444), 1.12 MiB | 4.59 MiB/s, done.
Resolving deltas: 100% (230/230), done.


In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

In [ ]:
hier_gen(submodules)


========== Iteration 0 / 10 ==========
Test           0 passed
Test           1 passed
Test           2 passed
Test           3 passed
Test 4 passed
Test 5 passed!
./mux2_1tb.v:60: $finish called at 60000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  4.605654001235962
Generation time:  4.479838609695435
Error handling time:  0.12581491470336914
Module  1  done:  2-to-1 multiplexer

========== Iteration 0 / 10 ==========
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed!
./mux4_1tb.v:40: $finish called at 90000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.7383434772491455
Generation time:  2.7227649688720703
Error handling time:  0.015578031539916992
Module  2  done:  4-to-1 multiplexer

========== Iteration 0 / 10 ==========
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 5 

(np.float64(23.331648111343384),
 np.float64(23.08522057533264),
 np.float64(0.24642467498779297))

In [ ]:
#@title Run PPA optimizer on hierarchical mux design
import os
import json

submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "mux2_1":  "./mux2_1tb.v",
    "mux4_1":  "./mux4_1tb.v",
    "mux8_1":  "./mux8_1tb.v",
    "mux16_1": "./mux16_1tb.v",
    "mux32_1": "./mux32_1tb.v",
    "mux64_1": "./mux64_1tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./mux_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="ChatGPT",
        clk_signal=None,   # all of these mux modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,         # set to 0 if you want less diagnostic output
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        # Fallback to the last generated module file in that run directory.
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path


Optimizing: mux2_1 - 2-to-1 multiplexer
Using prev_modules:

========== PPA Iteration 0 / 10 ==========
Iteration 0: Accepted new best candidate
  Candidate metrics: transistor_est=12, num_cells=1, wire_bits=4
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 1 / 10 ==========
Iteration 1: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 2 / 10 ==========
Iteration 2: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 3 / 10 ==========
Iteration 3: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7


In [ ]:
!apt-get install -y covered

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tklib
The following NEW packages will be installed:
  covered tklib
0 upgraded, 2 newly installed, 0 to remove and 120 not upgraded.
Need to get 1,996 kB of archives.
After this operation, 9,730 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 tklib all 0.8~20230222-1 [1,407 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 covered amd64 0.7.10-4 [588 kB]
Fetched 1,996 kB in 2s (843 kB/s)
Selecting previously unselected package tklib.
(Reading database ... 128409 files and directories currently installed.)
Preparing to unpack .../tklib_0.8~20230222-1_all.deb ...
Unpacking tklib (0.8~20230222-1) ...
Selecting previously unselected package covered.
Preparing to unpack .../covered_0.7.10-4_amd64.deb ...
Unpacking covered (0.7.10-4) ...
Setting up tklib (0.8~20230222-1) ...


Running Covered on mux64_1

In [ ]:
#@title Coverage helpers: iVerilog/vvp + Covered
import os, re, json, shutil, subprocess, time

def _sh(cmd, timeout=600):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or ""), (r.stderr or "")

def _sim_passed(text):
    low = (text or "").lower()
    return ("passed!" in low) and ("failed" not in low) and ("fatal" not in low)

def count_tests(tb_path, call_regex=r"^\s*check\s*\("):
    with open(tb_path) as f:
        return len(re.findall(call_regex, f.read(), flags=re.MULTILINE))

def run_coverage(top, tb_top, rtl_files, tb_file, run_dir, dut_inst="dut"):
    os.makedirs(run_dir, exist_ok=True)
    p = lambda name: os.path.join(run_dir, name)

    with open(p("rtl_all.v"), "w") as out:
        for f in rtl_files:
            out.write(open(f).read().rstrip() + "\n\n")
    shutil.copy(tb_file, p("tb_used.v"))

    # wrapper that turns on the VCD dump, so the TB itself is not edited
    with open(p("cov_dump.v"), "w") as f:
        f.write('module cov_dump;\n  initial begin\n    $dumpfile("%s");\n    $dumpvars(0, %s);\n  end\nendmodule\n'
                % (p("raw.vcd"), tb_top))

    rc, so, se = _sh(["iverilog", "-g2001", "-Wno-timescale", "-s", tb_top, "-s", "cov_dump",
                      "-o", p("sim.vvp"), p("rtl_all.v"), p("tb_used.v"), p("cov_dump.v")])
    if rc != 0:
        return {"ok": False, "stage": "compile", "sim_passed": False, "log": se + so}
    rc, so, se = _sh(["vvp", p("sim.vvp")])
    open(p("sim_output.txt"), "w").write(so + se)
    if rc != 0 or not _sim_passed(so + se):
        return {"ok": False, "stage": "simulate", "sim_passed": False, "log": so + se}

    # make the VCD readable by the old Covered 0.7.10 (same fix as the course demo)
    raw = open(p("raw.vcd")).read()
    raw = "".join(l for l in raw.splitlines(keepends=True) if not l.startswith("$comment "))
    raw = re.sub(r"\$timescale\s+(\S+)\s+\$end", r"$timescale \1 $end", raw)
    open(p("sim.vcd"), "w").write(raw)

    rc, so, se = _sh(["covered", "score", "-t", top, "-i", "%s.%s" % (tb_top, dut_inst),
                      "-v", p("rtl_all.v"), "-vcd", p("sim.vcd"), "-o", p("cov.cdd")])
    if rc != 0:
        return {"ok": False, "stage": "covered_score", "sim_passed": True, "log": so + se}
    _sh(["covered", "report", "-m", "ltc", "-d", "s", "-o", p("coverage_summary.rpt"), p("cov.cdd")])
    _sh(["covered", "report", "-m", "ltc", "-d", "v", "-o", p("coverage_detail.rpt"), p("cov.cdd")])

    res = parse_summary(open(p("coverage_summary.rpt")).read())
    res.update({"ok": True, "sim_passed": True, "run_dir": run_dir,
                "detail_rpt": p("coverage_detail.rpt"), "summary_rpt": p("coverage_summary.rpt")})
    return res

def parse_summary(text):
    out = {}
    sections = [("line", "LINE COVERAGE RESULTS"),
                ("toggle", "TOGGLE COVERAGE RESULTS"),
                ("logic", "COMBINATIONAL LOGIC COVERAGE RESULTS")]
    lines = text.splitlines()
    for key, header in sections:
        start = next(i for i, l in enumerate(lines) if header in l)
        row = next(l for l in lines[start:] if l.strip().startswith("Accumulated"))
        groups = re.findall(r"(\d+)/\s*(\d+)/\s*(\d+)\s+(\d+)%", row)
        if key == "toggle":
            out["toggle_0to1"] = int(groups[0][3]); out["toggle_1to0"] = int(groups[1][3])
            out["toggle_0to1_counts"] = list(map(int, groups[0][:3]))
            out["toggle_1to0_counts"] = list(map(int, groups[1][:3]))
        else:
            out[key] = int(groups[0][3]); out[key + "_counts"] = list(map(int, groups[0][:3]))
    return out

def record_history(history_path, design, label, tb_file, cov, rtl_note=""):
    data = {"design": design, "history": []}
    if os.path.exists(history_path):
        data = json.load(open(history_path))
    data["history"].append({
        "iteration": label, "num_tests": count_tests(tb_file),
        "line": cov["line"], "toggle_0to1": cov["toggle_0to1"],
        "toggle_1to0": cov["toggle_1to0"], "logic": cov["logic"],
        "counts": {k: cov[k] for k in cov if k.endswith("_counts")},
        "tb_file": tb_file, "run_dir": cov["run_dir"], "rtl_note": rtl_note,
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    })
    json.dump(data, open(history_path, "w"), indent=2)
    return data

def print_history(history_path):
    data = json.load(open(history_path))
    print("%-12s %7s %6s %11s %11s %6s" % ("Iteration", "#tests", "Line", "Tog 0->1", "Tog 1->0", "Logic"))
    for h in data["history"]:
        print("%-12s %7d %5d%% %10d%% %10d%% %5d%%" % (h["iteration"], h["num_tests"], h["line"],
              h["toggle_0to1"], h["toggle_1to0"], h["logic"]))

In [ ]:
mux_rtl = ["./mux%s_1/mux%s_1.v" % (n, n) for n in (2, 4, 8, 16, 32, 64)]
cov = run_coverage("mux64_1", "tb_mux64_1", mux_rtl, "./mux64_1tb.v", "./coverage_runs/mux64_1", dut_inst="dut")
print({k: v for k, v in cov.items() if k in ("line","toggle_0to1","toggle_1to0","logic","stage")})

{'line': 28, 'toggle_0to1': 100, 'toggle_1to0': 100, 'logic': 24}


In [ ]:
s = open(cov["summary_rpt"]).read()
i = s.index("LINE COVERAGE RESULTS"); print(s[i:i+1800])
d = open(cov["detail_rpt"]).read()
j = d.index("Missed Lines"); print(d[j:j+1500])

LINE COVERAGE RESULTS   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Module/Task/Function      Filename                 Hit/ Miss/Total    Percent hit
---------------------------------------------------------------------------------------------------------------------
  $root                   NA                         0/    0/    0      100%
  mux64_1                 rtl_all.v                  2/    0/    2      100%
  mux32_1                 rtl_all.v                  8/   26/   34       24%
---------------------------------------------------------------------------------------------------------------------
  Accumulated                                       10/   26/   36       28%


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   TOGGLE COVERAGE RES

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/HW_ROME_backup
!cp -r mux2_1 mux4_1 mux8_1 mux16_1 mux32_1 mux64_1 mux_ppa_runs coverage_runs half_adder full_adder adder4 adder8 adder8_repair* *tb.v /content/drive/MyDrive/HW_ROME_backup/ 2>/dev/null

Mounted at /content/drive


**Part II: Extend ROME to a new hierarchichal RTL design**

In [ ]:
#@title Feedback Loop

import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []
    for p in file_list:
        if not p:
            continue

        try:
            with open(p, "r") as f:
                text = f.read()
                buf.append(text)
                if not text.endswith("\n"):
                    buf.append("\n")
        except FileNotFoundError:
            print(f"[Warning] Previous module file not found and will be skipped: {p}")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the end of long logs, where compiler errors usually appear."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    tail = s[-max_chars:]
    return "[...truncated...]\n" + tail


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a way that works for API models and local HF models.

    ChatGPT/Gemini/HuggingFace can use a system role.
    Claude path in the earlier utility cell maps unsupported roles to user,
    but we keep Claude as user here to avoid role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for your Verilog testbenches.

    Accepts:
      - "All tests passed"
      - "passed!"
      - "Test x passed"

    Rejects:
      - "failed"
      - "Some tests failed"
      - non-empty fatal/error-like stderr
    """
    out = (stdout_text or "")
    err = (stderr_text or "")
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "error running",
        "fatal",
        "syntax error",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    outdir="",
    log=None,
    prev_modules=None,
    model_id="",
    seed_module_text=None,
    seed_feedback=""
):
    """
    Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Key behaviors:
      - The LLM is asked to output ONLY the current module.
      - For compilation, this concatenates:
            [prev_modules...] + [current module]
        into a temporary combined file.
      - After an error, repair iterations only resend:
            (a) previous module version
            (b) most recent compiler/simulation feedback
        to keep prompt size controlled.
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]
    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    if outdir is None:
        outdir = ""

    outdir = outdir.strip()

    if outdir != "":
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir == "":
            return filename
        return os.path.join(outdir, filename)

    # Normalize prev_modules into a list of paths.
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read the testbench so the model can target the test expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is incorrect or has a compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete corrected Verilog module, from module declaration through endmodule. "
        "Use Verilog-2001 only unless explicitly told otherwise."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench and design prompt."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it. Only use it to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: only previous module plus most recent error feedback."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module. "
            "Do not output explanations. "
            "Do not output a testbench. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)
        return conv

    if seed_module_text:
      conv = _new_conv_repair_only(seed_module_text, seed_feedback)
    else:
      conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    module_file = _outpath(module + ".v")
    combined_file = _outpath(module + "__combined.v")
    sim_output_file = _outpath(module)

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        print(f"\n========== Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        # Generate or repair only the current module.
        response = generate_verilog(conv, model_type, model_id=model_id)

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        conv.add_message("assistant", response)

        # Write raw model response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_" + str(iterations) + ".txt")
        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and write just the current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            last_module_text = response
            conv = _new_conv_repair_only(last_module_text, last_error_text)

            if iterations >= max_iterations:
                timeout = True

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)
            continue

        # Build concatenated source for compilation:
        # previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(module_file, "r") as f:
            curr_text = f.read()

        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)
            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # Compile with iVerilog.
        compile_cmd = [
            "iverilog",
            "-o",
            sim_output_file,
            combined_file,
            testbench,
        ]

        proc = subprocess.run(
            compile_cmd,
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            last_error_text = (
                "iverilog failed.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        elif proc.stderr.strip() != "":
            # Treat warnings as repair feedback.
            # This keeps generated modules cleaner for later synthesis/PPA.
            status = "Warnings compiling testbench"
            print(status)

            last_error_text = (
                "iverilog produced warnings.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        else:
            # Run simulation.
            run_cmd = ["vvp", sim_output_file]

            proc = subprocess.run(
                run_cmd,
                capture_output=True,
                text=True,
            )

            result_stdout = proc.stdout or ""
            result_stderr = proc.stderr or ""

            print(result_stdout)

            if proc.returncode != 0:
                status = "Error running testbench"
                print(status)

                last_error_text = (
                    "vvp simulation failed.\n\n"
                    "Command:\n"
                    + " ".join(run_cmd)
                    + "\n\nstdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            elif not _simulation_passed(result_stdout, result_stderr):
                status = "Testbench did not report success"
                print(status)

                last_error_text = (
                    "Simulation completed, but the output did not indicate success.\n\n"
                    "stdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            else:
                status = "Testbench ran successfully"
                print(status)

                last_error_text = ""
                success = True

        # Write bounded iteration log.
        with open(_outpath("log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

            if last_error_text:
                file.write("\nMost recent feedback:\n")
                file.write(_truncate_text(last_error_text))

        if not success:
            # Keep future repair prompts small.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("\n========== Feedback Loop Complete ==========")
    print("Final status: ", status)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return (
        np.sum(timelist_total),
        np.sum(timelist_gen),
        np.sum(timelist_error),
    )

In [ ]:
#@title Coverage helpers: iVerilog/vvp + Covered
import os, re, json, shutil, subprocess, time

def _sh(cmd, timeout=600):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or ""), (r.stderr or "")

def _sim_passed(text):
    low = (text or "").lower()
    return ("passed!" in low) and ("failed" not in low) and ("fatal" not in low)

def count_tests(tb_path, call_regex=r"^\s*check\s*\("):
    with open(tb_path) as f:
        return len(re.findall(call_regex, f.read(), flags=re.MULTILINE))

def run_coverage(top, tb_top, rtl_files, tb_file, run_dir, dut_inst="dut"):
    os.makedirs(run_dir, exist_ok=True)
    p = lambda name: os.path.join(run_dir, name)

    with open(p("rtl_all.v"), "w") as out:
        for f in rtl_files:
            out.write(open(f).read().rstrip() + "\n\n")
    shutil.copy(tb_file, p("tb_used.v"))

    # wrapper that turns on the VCD dump, so the TB itself is not edited
    with open(p("cov_dump.v"), "w") as f:
        f.write('module cov_dump;\n  initial begin\n    $dumpfile("%s");\n    $dumpvars(0, %s);\n  end\nendmodule\n'
                % (p("raw.vcd"), tb_top))

    rc, so, se = _sh(["iverilog", "-g2001", "-Wno-timescale", "-s", tb_top, "-s", "cov_dump",
                      "-o", p("sim.vvp"), p("rtl_all.v"), p("tb_used.v"), p("cov_dump.v")])
    if rc != 0:
        return {"ok": False, "stage": "compile", "sim_passed": False, "log": se + so}
    rc, so, se = _sh(["vvp", p("sim.vvp")])
    open(p("sim_output.txt"), "w").write(so + se)
    if rc != 0 or not _sim_passed(so + se):
        return {"ok": False, "stage": "simulate", "sim_passed": False, "log": so + se}

    # make the VCD readable by the old Covered 0.7.10 (same fix as the course demo)
    raw = open(p("raw.vcd")).read()
    raw = "".join(l for l in raw.splitlines(keepends=True) if not l.startswith("$comment "))
    raw = re.sub(r"\$timescale\s+(\S+)\s+\$end", r"$timescale \1 $end", raw)
    open(p("sim.vcd"), "w").write(raw)

    rc, so, se = _sh(["covered", "score", "-t", top, "-i", "%s.%s" % (tb_top, dut_inst),
                      "-v", p("rtl_all.v"), "-vcd", p("sim.vcd"), "-o", p("cov.cdd")])
    if rc != 0:
        return {"ok": False, "stage": "covered_score", "sim_passed": True, "log": so + se}
    _sh(["covered", "report", "-m", "ltc", "-d", "s", "-o", p("coverage_summary.rpt"), p("cov.cdd")])
    _sh(["covered", "report", "-m", "ltc", "-d", "v", "-o", p("coverage_detail.rpt"), p("cov.cdd")])

    res = parse_summary(open(p("coverage_summary.rpt")).read())
    res.update({"ok": True, "sim_passed": True, "run_dir": run_dir,
                "detail_rpt": p("coverage_detail.rpt"), "summary_rpt": p("coverage_summary.rpt")})
    return res

def parse_summary(text):
    out = {}
    sections = [("line", "LINE COVERAGE RESULTS"),
                ("toggle", "TOGGLE COVERAGE RESULTS"),
                ("logic", "COMBINATIONAL LOGIC COVERAGE RESULTS")]
    lines = text.splitlines()
    for key, header in sections:
        start = next(i for i, l in enumerate(lines) if header in l)
        row = next(l for l in lines[start:] if l.strip().startswith("Accumulated"))
        groups = re.findall(r"(\d+)/\s*(\d+)/\s*(\d+)\s+(\d+)%", row)
        if key == "toggle":
            out["toggle_0to1"] = int(groups[0][3]); out["toggle_1to0"] = int(groups[1][3])
            out["toggle_0to1_counts"] = list(map(int, groups[0][:3]))
            out["toggle_1to0_counts"] = list(map(int, groups[1][:3]))
        else:
            out[key] = int(groups[0][3]); out[key + "_counts"] = list(map(int, groups[0][:3]))
    return out

def record_history(history_path, design, label, tb_file, cov, rtl_note=""):
    data = {"design": design, "history": []}
    if os.path.exists(history_path):
        data = json.load(open(history_path))
    data["history"].append({
        "iteration": label, "num_tests": count_tests(tb_file),
        "line": cov["line"], "toggle_0to1": cov["toggle_0to1"],
        "toggle_1to0": cov["toggle_1to0"], "logic": cov["logic"],
        "counts": {k: cov[k] for k in cov if k.endswith("_counts")},
        "tb_file": tb_file, "run_dir": cov["run_dir"], "rtl_note": rtl_note,
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    })
    json.dump(data, open(history_path, "w"), indent=2)
    return data

def print_history(history_path):
    data = json.load(open(history_path))
    print("%-12s %7s %6s %11s %11s %6s" % ("Iteration", "#tests", "Line", "Tog 0->1", "Tog 1->0", "Logic"))
    for h in data["history"]:
        print("%-12s %7d %5d%% %10d%% %10d%% %5d%%" % (h["iteration"], h["num_tests"], h["line"],
              h["toggle_0to1"], h["toggle_1to0"], h["logic"]))

In [ ]:
#@title Coverage targets: detailed Covered report to LLM feedback
import re

def parse_detail(text):
    lines = text.splitlines()
    section, module = None, None
    missed_lines, toggles, logic = [], [], []
    i = 0
    while i < len(lines):
        l = lines[i]
        if "LINE COVERAGE RESULTS" in l:                  section = "line"
        elif "TOGGLE COVERAGE RESULTS" in l:              section = "toggle"
        elif "COMBINATIONAL LOGIC COVERAGE RESULTS" in l: section = "logic"
        m = re.search(r"Module: (\S+?), File:", l)
        if m:
            module = m.group(1)

        if section == "line" and module:
            m = re.match(r"\s+(\d+):\s+(.*\S)\s*$", l)
            if m:
                missed_lines.append((module, int(m.group(1)), re.sub(r"\s+", " ", m.group(2))))

        # toggle: the hex value is the set of bits that DID toggle
        if section == "toggle" and module:
            m = re.match(r"\s+(\S+)\s+0->1:\s+(\d+)'h([0-9a-fA-F]+)", l)
            if m and i + 1 < len(lines):
                m2 = re.search(r"1->0:\s+(\d+)'h([0-9a-fA-F]+)", lines[i + 1])
                if m2:
                    sig, width = m.group(1), int(m.group(2))
                    up, down = int(m.group(3), 16), int(m2.group(2), 16)
                    miss_up = [b for b in range(width) if not (up >> b) & 1]
                    miss_dn = [b for b in range(width) if not (down >> b) & 1]
                    toggles.append((module, sig, width, miss_up, miss_dn))

        if section == "logic" and module and re.match(r"\s*=[^=|]+=(\|=[^=|]+=)*\s*$", l) and "=====" not in l:
            star = lines[i + 1] if i + 1 < len(lines) else ""
            missed = []
            for m in re.finditer(r"=([^=|]+)=", l):
                if "*" in star[m.start():m.end()]:
                    missed.append(m.group(1))
            src_no, src_txt, op = None, "", ""
            for j in range(i - 1, max(i - 12, 0), -1):
                mm = re.match(r"\s+(\d+):\s+(.*\S)\s*$", lines[j])
                if mm:
                    src_no, src_txt = int(mm.group(1)), re.sub(r"\s+", " ", mm.group(2)); break
            for j in range(i - 1, max(i - 6, 0), -1):
                mo = re.match(r"\s*\^+\s+-\s*(.*)$", lines[j])
                if mo:
                    op = mo.group(1).strip(); break
            if missed:
                logic.append((module, src_no, src_txt, op, missed))
        i += 1
    return missed_lines, toggles, logic

def build_feedback(detail_path, cov, max_items=25):
    ml, tg, lg = parse_detail(open(detail_path).read())
    out = []
    out.append("Current coverage (accumulated over the hierarchy): line %d%%, toggle 0->1 %d%%, toggle 1->0 %d%%, logic %d%%."
               % (cov["line"], cov["toggle_0to1"], cov["toggle_1to0"], cov["logic"]))
    if ml:
        out.append("\nUNCOVERED RTL LINES (never executed):")
        out += ["- module %s, line %d: %s" % m for m in ml[:max_items]]
    if tg:
        out.append("\nSIGNALS WITH MISSING TOGGLE DIRECTIONS (bit index = position in the vector):")
        for mod, sig, w, up, dn in tg[:max_items]:
            parts = []
            if up: parts.append("never rises 0->1 on bits %s" % up)
            if dn: parts.append("never falls 1->0 on bits %s" % dn)
            out.append("- module %s, signal %s[%d]: %s" % (mod, sig, w, "; ".join(parts)))
    if lg:
        out.append("\nUNCOVERED LOGIC COMBINATIONS (L,R = left/right operand of the operator; '-' = don't care):")
        for mod, no, txt, op, missed in lg[:max_items]:
            if op and not op.startswith("["):
                out.append("- module %s, line %s: `%s` operator '%s' missing input combos: %s"
                           % (mod, no, txt, op, ", ".join(missed)))
            else:
                out.append("- module %s, line %s: `%s` signal never took value(s): %s"
                           % (mod, no, txt, ", ".join(missed)))
    out.append("\nNOTE: only the top-level module's inputs can be driven by the testbench. Signals inside submodules "
               "(e.g. carries) are reached by choosing top-level input patterns.")
    return "\n".join(out)

In [ ]:
#@title Coverage targets: detailed Covered report to LLM feedback
import re

def parse_detail(text):
    lines = text.splitlines()
    section, module = None, None
    missed_lines, toggles, logic = [], [], []
    i = 0
    while i < len(lines):
        l = lines[i]
        if "LINE COVERAGE RESULTS" in l:                  section = "line"
        elif "TOGGLE COVERAGE RESULTS" in l:              section = "toggle"
        elif "COMBINATIONAL LOGIC COVERAGE RESULTS" in l: section = "logic"
        m = re.search(r"Module: (\S+?), File:", l)
        if m:
            module = m.group(1)

        if section == "line" and module:
            m = re.match(r"\s+(\d+):\s+(.*\S)\s*$", l)
            if m:
                missed_lines.append((module, int(m.group(1)), re.sub(r"\s+", " ", m.group(2))))

        # toggle: the hex value is the set of bits that DID toggle
        if section == "toggle" and module:
            m = re.match(r"\s+(\S+)\s+0->1:\s+(\d+)'h([0-9a-fA-F]+)", l)
            if m and i + 1 < len(lines):
                m2 = re.search(r"1->0:\s+(\d+)'h([0-9a-fA-F]+)", lines[i + 1])
                if m2:
                    sig, width = m.group(1), int(m.group(2))
                    up, down = int(m.group(3), 16), int(m2.group(2), 16)
                    miss_up = [b for b in range(width) if not (up >> b) & 1]
                    miss_dn = [b for b in range(width) if not (down >> b) & 1]
                    toggles.append((module, sig, width, miss_up, miss_dn))

        if section == "logic" and module and re.match(r"\s*=[^=|]+=(\|=[^=|]+=)*\s*$", l) and "=====" not in l:
            star = lines[i + 1] if i + 1 < len(lines) else ""
            missed = []
            for m in re.finditer(r"=([^=|]+)=", l):
                if "*" in star[m.start():m.end()]:
                    missed.append(m.group(1))
            src_no, src_txt, op = None, "", ""
            for j in range(i - 1, max(i - 12, 0), -1):
                mm = re.match(r"\s+(\d+):\s+(.*\S)\s*$", lines[j])
                if mm:
                    src_no, src_txt = int(mm.group(1)), re.sub(r"\s+", " ", mm.group(2)); break
            for j in range(i - 1, max(i - 6, 0), -1):
                mo = re.match(r"\s*\^+\s+-\s*(.*)$", lines[j])
                if mo:
                    op = mo.group(1).strip(); break
            if missed:
                logic.append((module, src_no, src_txt, op, missed))
        i += 1
    return missed_lines, toggles, logic

def build_feedback(detail_path, cov, max_items=25):
    ml, tg, lg = parse_detail(open(detail_path).read())
    out = []
    out.append("Current coverage (accumulated over the hierarchy): line %d%%, toggle 0->1 %d%%, toggle 1->0 %d%%, logic %d%%."
               % (cov["line"], cov["toggle_0to1"], cov["toggle_1to0"], cov["logic"]))
    if ml:
        out.append("\nUNCOVERED RTL LINES (never executed):")
        out += ["- module %s, line %d: %s" % m for m in ml[:max_items]]
    if tg:
        out.append("\nSIGNALS WITH MISSING TOGGLE DIRECTIONS (bit index = position in the vector):")
        for mod, sig, w, up, dn in tg[:max_items]:
            parts = []
            if up: parts.append("never rises 0->1 on bits %s" % up)
            if dn: parts.append("never falls 1->0 on bits %s" % dn)
            out.append("- module %s, signal %s[%d]: %s" % (mod, sig, w, "; ".join(parts)))
    if lg:
        out.append("\nUNCOVERED LOGIC COMBINATIONS (L,R = left/right operand of the operator; '-' = don't care):")
        for mod, no, txt, op, missed in lg[:max_items]:
            if op and not op.startswith("["):
                out.append("- module %s, line %s: `%s` operator '%s' missing input combos: %s"
                           % (mod, no, txt, op, ", ".join(missed)))
            else:
                out.append("- module %s, line %s: `%s` signal never took value(s): %s"
                           % (mod, no, txt, ", ".join(missed)))
    out.append("\nNOTE: only the top-level module's inputs can be driven by the testbench. Signals inside submodules "
               "(e.g. carries) are reached by choosing top-level input patterns.")
    return "\n".join(out)

In [ ]:
#@title Coverage loop: TB refinement only with RTL fixed & RTL repair wrapper with fixed TB
import os, re

TB_BEGIN = "// COVERAGE_TESTS_BEGIN"
TB_END   = "// COVERAGE_TESTS_END"

def _strip_fences(s):
    s = (s or "").strip()
    s = re.sub(r"^```[a-zA-Z]*\s*\n", "", s)
    s = re.sub(r"\n```\s*$", "", s)
    return s.strip("\n")

def refine_testbench(tb_in, tb_out, feedback, model_type=None, model_id=""):
    """TESTBENCH-ONLY step. RTL is never touched. Existing checks are preserved."""
    tb = open(tb_in).read()
    if TB_BEGIN not in tb or TB_END not in tb:
        raise ValueError("Testbench needs the markers %s and %s inside its main initial block." % (TB_BEGIN, TB_END))

    prompt = (
        "You are extending an existing self-checking Verilog-2001 testbench with targeted tests.\n"
        "Do not rewrite the testbench. Do not remove or change any existing check.\n"
        "Output only new Verilog statements that will be pasted inside the existing initial block, right before the line\n"
        "'" + TB_END + "'. Rules:\n"
        "- Use only signals, variables and tasks that are already declared in the testbench (for example call the existing check task).\n"
        "- Do not output module/endmodule, initial, $finish, $dumpfile/$dumpvars, declarations, or the final pass/fail $display.\n"
        "- Expected values must come from the testbench's independent reference expression, never from the DUT.\n"
        "- Aim every new test at a specific uncovered target below. Keep it to about 3-8 new tests.\n"
        "- No Markdown fences, no explanations; Verilog statements only (comments are fine).\n\n"
        "CURRENT TESTBENCH:\n```verilog\n" + tb + "\n```\n\n"
        "UNCOVERED TARGETS FROM COVERED:\n" + feedback + "\n"
    )
    conv = Conversation(log_file=tb_out + ".llm_log.txt")
    conv.add_message("user", prompt)
    reply = _strip_fences(generate_verilog(conv, model_type, model_id=model_id))

    code_only = re.sub(r"//.*", "", reply)          # comments are allowed to mention anything
    banned = ["endmodule", "$finish", "$dumpfile", "$dumpvars", "passed!"]
    bad = [b for b in banned if b in code_only] + (["module"] if re.search(r"\bmodule\b", code_only) else [])
    if bad:
        raise ValueError("LLM reply contains forbidden constructs %s; rerun. Reply was:\n%s" % (bad, reply))

    body = "\n".join("    " + l if l.strip() else l for l in reply.splitlines())
    new_tb = tb.replace(TB_END, body + "\n    " + TB_END, 1)
    open(tb_out, "w").write(new_tb)
    open(tb_out + ".added_tests.txt", "w").write(reply + "\n")
    return reply

def coverage_loop(top, tb_top, rtl_files, tb_start, work_dir, history_path,
                  start_round=0, max_round=2, model_type=None, model_id="", dut_inst="dut"):
    """round 0 = Baseline, round k = 'Refine k'. Stops with status='rtl_fail' if a TB fails on the current RTL."""
    os.makedirs(work_dir, exist_ok=True)
    cur_tb, r = tb_start, start_round
    while True:
        label = "Baseline" if r == 0 else "Refine %d" % r
        run_dir = os.path.join(work_dir, "cov_" + label.replace(" ", "_").lower())
        print("\n=== %s: TB=%s ===" % (label, cur_tb))
        cov = run_coverage(top, tb_top, rtl_files, cur_tb, run_dir, dut_inst=dut_inst)
        if not cov["ok"]:
            print("TB does not pass / cannot be scored at stage '%s'. Freeze this TB and repair the RTL." % cov["stage"])
            print(cov["log"][-2000:])
            return {"status": "rtl_fail", "frozen_tb": cur_tb, "round": r, "log": cov["log"]}
        record_history(history_path, top, label, cur_tb, cov)
        print("line %d%% | toggle 0->1 %d%% | toggle 1->0 %d%% | logic %d%%"
              % (cov["line"], cov["toggle_0to1"], cov["toggle_1to0"], cov["logic"]))
        if r >= max_round:
            return {"status": "done", "final_tb": cur_tb, "round": r}
        fb = build_feedback(cov["detail_rpt"], cov)
        open(os.path.join(run_dir, "feedback_for_llm.txt"), "w").write(fb)
        print("\n--- feedback sent to LLM ---\n" + fb)
        next_tb = os.path.join(work_dir, "tb_iter_%d.v" % (r + 1))
        added = refine_testbench(cur_tb, next_tb, fb, model_type, model_id)
        print("\n--- tests added in %s ---\n%s" % (next_tb, added))
        cur_tb, r = next_tb, r + 1

def repair_rtl_with_frozen_tb(module, rtl_files, frozen_tb, design_prompt, failure_log,
                              outdir, model_type=None, model_id="", max_iterations=10):
    """RTL-ONLY step: ROME's verilog_loop with the stronger TB held fixed. Returns the new rtl_files list."""
    cur = next(f for f in rtl_files if os.path.basename(f) == module + ".v")
    others = [f for f in rtl_files if f != cur]
    verilog_loop(design_prompt, module, frozen_tb, max_iterations, model_type,
                 outdir=outdir, log=os.path.join(outdir, "log.txt"),
                 prev_modules=others, model_id=model_id,
                 seed_module_text=open(cur).read(), seed_feedback=failure_log)
    return others + [os.path.join(outdir, module + ".v")]

In [ ]:
%%writefile half_addertb.v
module half_adder_tb;
  reg a, b; wire sum, carry;
  integer errors;
  half_adder dut(.a(a), .b(b), .sum(sum), .carry(carry));
  task check(input ta, input tb_);
    begin
      a = ta; b = tb_; #1;
      if ({carry, sum} !== ta + tb_) begin
        errors = errors + 1;
        $display("FAILED half_adder a=%b b=%b got carry=%b sum=%b", ta, tb_, carry, sum);
      end
    end
  endtask
  initial begin
    errors = 0;
    check(0, 0);
    check(0, 1);
    check(1, 0);
    check(1, 1);
    if (errors == 0) $display("half_adder passed!"); else $display("half_adder Failed");
    $finish;
  end
endmodule

Overwriting half_addertb.v


In [ ]:
%%writefile full_addertb.v
module full_adder_tb;
  reg a, b, cin; wire sum, cout;
  integer errors, i;
  full_adder dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
  initial begin
    errors = 0;
    for (i = 0; i < 8; i = i + 1) begin
      {a, b, cin} = i[2:0]; #1;
      if ({cout, sum} !== a + b + cin) begin
        errors = errors + 1;
        $display("FAILED full_adder a=%b b=%b cin=%b got cout=%b sum=%b", a, b, cin, cout, sum);
      end
    end
    if (errors == 0) $display("full_adder passed!"); else $display("full_adder failed");
    $finish;
  end
endmodule

Overwriting full_addertb.v


In [ ]:
%%writefile adder4tb.v
module adder4_tb;
  reg [3:0] a, b; reg cin; wire [3:0] sum; wire cout;
  integer errors;
  adder4 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
  task check(input [3:0] ta, input [3:0] tb_, input tc);
    reg [4:0] expv;
    begin
      a = ta; b = tb_; cin = tc; #1;
      expv = ta + tb_ + tc;
      if ({cout, sum} !== expv) begin
        errors = errors + 1;
        $display("FAILED adder4 a=%0d b=%0d cin=%0d got=%0d exp=%0d", ta, tb_, tc, {cout, sum}, expv);
      end
    end
  endtask
  initial begin
    errors = 0;
    check(4'd0, 4'd0, 1'b0);
    check(4'd1, 4'd2, 1'b0);
    check(4'd7, 4'd8, 1'b0);
    check(4'd9, 4'd9, 1'b1);
    check(4'd15, 4'd15, 1'b1);
    if (errors == 0) $display("adder4 passed!"); else $display("adder4 failed");
    $finish;
  end
endmodule

Overwriting adder4tb.v


In [ ]:
%%writefile adder8tb.v
module adder8_tb;
  reg [7:0] a, b; reg cin; wire [7:0] sum; wire cout;
  integer errors;
  adder8 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
  task check(input [7:0] ta, input [7:0] tb_, input tc);
    reg [8:0] expv;
    begin
      a = ta; b = tb_; cin = tc; #1;
      expv = ta + tb_ + tc;
      if ({cout,sum} !== expv) begin
        errors = errors + 1;
        $display("FAILED a=%0d b=%0d cin=%0d got=%0d exp=%0d", ta, tb_, tc, {cout,sum}, expv);
      end
    end
  endtask
  initial begin
    errors = 0;
    check(8'd1, 8'd2, 1'b0);
    check(8'd10, 8'd20, 1'b0);
    check(8'd0,  8'd0,  1'b0);
    // COVERAGE_TESTS_BEGIN
    // COVERAGE_TESTS_END
    if (errors == 0) $display("adder8 passed!"); else $display("adder8 failed");
    $finish;
  end
endmodule

Overwriting adder8tb.v


In [ ]:
adder_modules = [
    ["half_adder", "half adder",
     "input a, input b, output sum, output carry"],
    ["full_adder", "full adder built from two half_adder instances",
     "input a, input b, input cin, output sum, output cout"],
    ["adder4", "4-bit ripple-carry adder built from four full_adder instances",
     "input [3:0] a, input [3:0] b, input cin, output [3:0] sum, output cout"],
    ["adder8", "8-bit adder built from two adder4 instances",
     "input [7:0] a, input [7:0] b, input cin, output [7:0] sum, output cout"],
]
hier_gen(adder_modules, testbench_template="./{module}tb.v")


========== Iteration 0 / 10 ==========
half_adder passed!
./half_addertb.v:21: $finish called at 4 (1s)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.5506439208984375
Generation time:  2.5266165733337402
Error handling time:  0.024026870727539062
Module  1  done:  half adder

========== Iteration 0 / 10 ==========
full_adder passed!
./full_addertb.v:15: $finish called at 8 (1s)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.119173288345337
Generation time:  2.0924861431121826
Error handling time:  0.026686429977416992
Module  2  done:  full adder built from two half_adder instances

========== Iteration 0 / 10 ==========
adder4 passed!
./adder4tb.v:24: $finish called at 5 (1s)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.8931474685668

(np.float64(10.306647777557373),
 np.float64(10.185606956481934),
 np.float64(0.12103796005249023))

In [ ]:
!mkdir -p /content/drive/MyDrive/HW_ROME_backup
!cp -r half_adder full_adder adder4 adder8 *tb.v /content/drive/MyDrive/HW_ROME_backup/

In [ ]:
RTL_FILES = ["./half_adder/half_adder.v", "./full_adder/full_adder.v",
             "./adder4/adder4.v", "./adder8/adder8.v"]
COV_DIR = "./coverage_runs/adder8"
HISTORY = COV_DIR + "/coverage_history.json"

result = coverage_loop(top="adder8", tb_top="adder8_tb", rtl_files=RTL_FILES,
                       tb_start="./adder8tb.v", work_dir=COV_DIR, history_path=HISTORY,
                       start_round=0, max_round=2, dut_inst="dut")
print(result["status"])
print_history(HISTORY)


=== Baseline: TB=./adder8tb.v ===
line 100% | toggle 0->1 38% | toggle 1->0 50% | logic 60%

--- feedback sent to LLM ---
Current coverage (accumulated over the hierarchy): line 100%, toggle 0->1 38%, toggle 1->0 50%, logic 60%.

SIGNALS WITH MISSING TOGGLE DIRECTIONS (bit index = position in the vector):
- module adder8, signal a[8]: never rises 0->1 on bits [0, 2, 4, 5, 6, 7]; never falls 1->0 on bits [2, 4, 5, 6, 7]
- module adder8, signal b[8]: never rises 0->1 on bits [0, 1, 3, 5, 6, 7]; never falls 1->0 on bits [0, 3, 5, 6, 7]
- module adder8, signal cin[1]: never rises 0->1 on bits [0]; never falls 1->0 on bits [0]
- module adder8, signal sum[8]: never rises 0->1 on bits [0, 1, 5, 6, 7]; never falls 1->0 on bits [5, 6, 7]
- module adder8, signal cout[1]: never rises 0->1 on bits [0]; never falls 1->0 on bits [0]
- module adder8, signal c4[1]: never rises 0->1 on bits [0]; never falls 1->0 on bits [0]
- module adder4, signal a[4]: never rises 0->1 on bits [0, 2]; never falls 1->

In [ ]:
!cp -r coverage_runs /content/drive/MyDrive/HW_ROME_backup/

In [ ]:
import subprocess, shutil
final_tb = result["final_tb"]
shutil.copy(final_tb, "./tb_final.v")
cmds = ["iverilog -g2001 -s adder8_tb -o final_sim.vvp " + " ".join(RTL_FILES) + " tb_final.v",
        "vvp final_sim.vvp"]
for c in cmds:
    print("$", c)
    print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout)

$ iverilog -g2001 -s adder8_tb -o final_sim.vvp ./half_adder/half_adder.v ./full_adder/full_adder.v ./adder4/adder4.v ./adder8/adder8.v tb_final.v

$ vvp final_sim.vvp
adder8 passed!
tb_final.v:49: $finish called at 18 (1s)



In [ ]:
!rm -rf HW_ROME_Coverage && mkdir -p HW_ROME_Coverage/testbenches
!cp -r mux2_1 mux4_1 mux8_1 mux16_1 mux32_1 mux64_1 half_adder full_adder adder4 adder8 mux_ppa_runs coverage_runs HW_ROME_Coverage/ 2>/dev/null
!cp mux*tb.v half_addertb.v full_addertb.v adder4tb.v adder8tb.v tb_final.v HW_ROME_Coverage/testbenches/
!cp coverage_runs/adder8/tb_iter_1.v coverage_runs/adder8/tb_iter_2.v HW_ROME_Coverage/testbenches/
!cp coverage_runs/adder8/coverage_history.json HW_ROME_Coverage/
!cp coverage_runs/mux64_1/coverage_summary.rpt HW_ROME_Coverage/mux64_1_coverage_summary.rpt 2>/dev/null
!zip -qr HW_ROME_Coverage.zip HW_ROME_Coverage
from google.colab import files; files.download("HW_ROME_Coverage.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>